# Before vs After Fine-tuning — X-Rec Metric Evaluation

Uses the **exact functions** from `evaluation/metrics.py` to compare the original LLaMA base model with the refined LLaMA for counterfactual data.

| | File | Field |
|--|------|-------|
| **BEFORE** | `counterfactual_training_dataset.json` | `llama_explanation` |
| **AFTER** | `counterfactual_finetuned_results.json` | `llama_explanation` |
| **REF** | `counterfactual_finetuned_results.json` | `chatgpt_reference` |


In [6]:
# Cell 1: Install
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evaluate', 'bert_score', 'openai'], check=True)
print('OK')

OK


In [7]:
# Cell 2: Build aligned lists + save pkl files
import json, pickle, os

# ── Paths (change to /content/... for Colab) ──────────────────────────────────
FINETUNED_JSON = 'counterfactual_finetuned_results.json'
ORIGINAL_JSON  = 'counterfactual_training_dataset.json'
OUT_DIR        = 'eval_pkls'
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

# Load both JSONs
with open(FINETUNED_JSON) as f:
    ft_data = json.load(f)
with open(ORIGINAL_JSON) as f:
    orig_data = json.load(f)

# Build lookup: user_id -> original llama_explanation
orig_lookup = {}
for r in orig_data:
    uid = r['user_id']
    if uid not in orig_lookup:  # keep first occurrence
        orig_lookup[uid] = r.get('llama_explanation', '').strip()

# Build aligned lists using finetuned file as the index
before_preds = []
after_preds  = []
refs         = []
skipped      = 0

for r in ft_data:
    uid   = r['user_id']
    after = r.get('llama_explanation', '').strip()
    ref   = r.get('chatgpt_reference', '').strip()
    before = orig_lookup.get(uid, '').strip()

    # Skip if any field is empty
    if not after or not ref or not before:
        skipped += 1
        continue

    before_preds.append(before)
    after_preds.append(after)
    refs.append(ref)

print(f'Fine-tuned records  : {len(ft_data)}')
print(f'Original records    : {len(orig_data)} (unique user_ids: {len(orig_lookup)})')
print(f'Aligned records     : {len(before_preds)}')
print(f'Skipped (empty)     : {skipped}')

# Save pkl files
with open(f'{OUT_DIR}/before_tst_pred.pkl', 'wb') as f: pickle.dump(before_preds, f)
with open(f'{OUT_DIR}/after_tst_pred.pkl',  'wb') as f: pickle.dump(after_preds, f)
with open(f'{OUT_DIR}/tst_ref.pkl',          'wb') as f: pickle.dump(refs, f)

print(f'\nSaved to {OUT_DIR}/')
print(f'  before_tst_pred.pkl  ({len(before_preds)} strings)')
print(f'  after_tst_pred.pkl   ({len(after_preds)} strings)')
print(f'  tst_ref.pkl          ({len(refs)} strings)')

Fine-tuned records  : 2200
Original records    : 5668 (unique user_ids: 5668)
Aligned records     : 2199
Skipped (empty)     : 1

Saved to eval_pkls/
  before_tst_pred.pkl  (2199 strings)
  after_tst_pred.pkl   (2199 strings)
  tst_ref.pkl          (2199 strings)


In [8]:
# Cell 3: Sanity check — show 2 sample comparisons
for i in [0, 1]:
    print(f'=== Record {i} ===')
    print(f'BEFORE : {before_preds[i][:140]}')
    print(f'AFTER  : {after_preds[i][:140]}')
    print(f'REF    : {refs[i][:140]}')
    print()

=== Record 0 ===
BEFORE : the user is drawn to books that delve into human nature and provide insightful observations, but they seem uninterested in post-war themes o
AFTER  : Your profile suggests that Collapse is not strongly supported as a recommendation. Your profile emphasizes Based on the user's purchases and
REF    : Based on the patterns in your profile, the system would not recommend Collapse. Your profile emphasizes Based on the user's purchases and re

=== Record 1 ===
BEFORE : the user's enjoyment of "Giving It Up" would increase if they were to explore more books like Dirty Aristocrat and One Insatiable, as well a
AFTER  : When this recommendation is evaluated against your profile, Giving It Up (The Lost Girls) remains below recommendable strength for you. Your
REF    : From the evidence in your preferences, Giving It Up (The Lost Girls) remains an uncertain fit rather than a recommendation. Your profile emp



In [9]:
# Cell 4: USR (Unique Sentence Ratio)
# By calculating the percentage of distinct sequences or words
# in the total batch, this cell determines the diversity of the
# explanations that were produced.
import numpy as np

def two_seq_same(sa, sb):
    """Helper to check if two token sequences are exactly identical."""
    if len(sa) != len(sb): return False
    return all(a == b for a, b in zip(sa, sb))

def unique_sentence_percent(sequence_batch):
    """
    Calculates the ratio of unique sequences in a batch.
    Returns the ratio (0.0 to 1.0) and the absolute number of unique sequences.
    """
    unique_seq = []
    for seq in sequence_batch:
        count = sum(1 for u in unique_seq if two_seq_same(seq, u))
        if count == 0:
            unique_seq.append(seq)
    return len(unique_seq) / len(sequence_batch), len(unique_seq)

# Tokenize the predictions (splitting by whitespace)
before_tokens = [s.split() for s in before_preds]
after_tokens  = [s.split() for s in after_preds]

# Calculate USR for both 'before' and 'after' sets
usr_before, n_before = unique_sentence_percent(before_tokens)
usr_after,  n_after  = unique_sentence_percent(after_tokens)

print(f'USR  BEFORE: {usr_before:.4f}  ({n_before}/{len(before_preds)} unique)')
print(f'USR  AFTER : {usr_after:.4f}  ({n_after}/{len(after_preds)} unique)')
print(f'Delta      : {usr_after - usr_before:+.4f}')

USR  BEFORE: 0.9845  (2165/2199 unique)
USR  AFTER : 1.0000  (2199/2199 unique)
Delta      : +0.0155


In [ ]:
# Cell 5: BERTScore
# BERTScore evaluates the semantic similarity between the generated explanations
# and the reference explanations using pre-trained contextual embeddings from BERT.
# Testing 300 samples.
import evaluate, random

SAMPLE = 300   # set to len(before_preds) for full run (slow on CPU)
random.seed(42)
idx = random.sample(range(len(before_preds)), min(SAMPLE, len(before_preds)))

s_before = [before_preds[i] for i in idx]
s_after  = [after_preds[i]  for i in idx]
s_ref    = [refs[i]          for i in idx]

bertscore = evaluate.load('bertscore')

print(f'Computing BERTScore on {len(idx)} samples (BEFORE)...')
bs_before = bertscore.compute(predictions=s_before, references=s_ref,
                               lang='en', rescale_with_baseline=True)
print(f'Computing BERTScore on {len(idx)} samples (AFTER)...')
bs_after  = bertscore.compute(predictions=s_after,  references=s_ref,
                               lang='en', rescale_with_baseline=True)

bf1  = np.mean(bs_before['f1']);   af1  = np.mean(bs_after['f1'])
bp   = np.mean(bs_before['precision']); ap   = np.mean(bs_after['precision'])
br   = np.mean(bs_before['recall']);    ar   = np.mean(bs_after['recall'])
bf1s = np.std(bs_before['f1']);    af1s = np.std(bs_after['f1'])

print()
print('=' * 68)
print(f'  BEFORE vs AFTER  (n={len(idx)} sample, CPU)')
print(f'  Reference = ChatGPT explanations')
print('=' * 68)
print(f'  {"Metric":<26} {"Before":>10} {"After":>10} {"Delta":>8}')
print('  ' + '-' * 56)
print(f'  {"BERTScore F1":<26} {bf1:>10.4f} {af1:>10.4f} {af1-bf1:>+8.4f}')
print(f'  {"BERTScore Precision":<26} {bp:>10.4f} {ap:>10.4f} {ap-bp:>+8.4f}')
print(f'  {"BERTScore Recall":<26} {br:>10.4f} {ar:>10.4f} {ar-br:>+8.4f}')
print(f'  {"BERTScore F1 std":<26} {bf1s:>10.4f} {af1s:>10.4f}')
print(f'  {"USR (full dataset)":<26} {usr_before:>10.4f} {usr_after:>10.4f} {usr_after-usr_before:>+8.4f}')
print('=' * 68)
print(f'  BERTScore F1  : {"improved" if af1 > bf1 else "did not improve"} ({af1-bf1:+.4f})')

Computing BERTScore on 300 samples (BEFORE)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Computing BERTScore on 300 samples (AFTER)...

  BEFORE vs AFTER  (n=300 sample, CPU)
  Reference = ChatGPT explanations
  Metric                         Before      After    Delta
  --------------------------------------------------------
  BERTScore F1                   0.0121     0.4516  +0.4395
  BERTScore Precision            0.2136     0.5438  +0.3302
  BERTScore Recall              -0.1764     0.3609  +0.5373
  BERTScore F1 std               0.0699     0.1130
  USR (full dataset)             0.9845     1.0000  +0.0155
  ✅ BERTScore F1 : improved (+0.4395)


---
## Cells 6–7: Full Dataset Evaluation in Colab

Before running these cells, ensure the following files are uploaded to the Colab `/content/` directory:
- `counterfactual_training_dataset.json` (Original data)
- `counterfactual_finetuned_results.json` (Finetuned data)
- `metrics.py` (Evaluation scripts from `evaluation/`)
- `system_prompt.txt` (Prompt definitions from `evaluation/`)

* **Cell 6:** Parses the uploaded JSON files to build aligned `.pkl` files for the "before", "after", and "reference" explanations, ensuring only complete records are evaluated.
* **Cell 7:** Executes the full dataset evaluation using the exact functions from `metrics.py`. It calculates the BERTScore using GPU acceleration and computes the Unique Sentence Ratio (diversity). It also patches the GPT scoring function to properly parse responses from the TAMU AI API.

In [11]:
# Cell 6: builds pkl files for GPT evaluations
import json, pickle, os

FINETUNED_JSON = '/content/counterfactual_finetuned_results.json'
ORIGINAL_JSON  = '/content/counterfactual_training_dataset.json'
OUT_DIR        = '/content/eval_pkls'

os.makedirs(OUT_DIR, exist_ok=True)

with open(FINETUNED_JSON) as f: ft_data   = json.load(f)
with open(ORIGINAL_JSON)  as f: orig_data = json.load(f)

orig_lookup = {}
for r in orig_data:
    uid = r['user_id']
    if uid not in orig_lookup:
        orig_lookup[uid] = r.get('llama_explanation', '').strip()

before_preds, after_preds, refs = [], [], []
for r in ft_data:
    uid    = r['user_id']
    after  = r.get('llama_explanation', '').strip()
    ref    = r.get('chatgpt_reference', '').strip()
    before = orig_lookup.get(uid, '').strip()
    if after and ref and before:
        before_preds.append(before)
        after_preds.append(after)
        refs.append(ref)

with open(f'{OUT_DIR}/before_tst_pred.pkl', 'wb') as f: pickle.dump(before_preds, f)
with open(f'{OUT_DIR}/after_tst_pred.pkl',  'wb') as f: pickle.dump(after_preds, f)
with open(f'{OUT_DIR}/tst_ref.pkl',          'wb') as f: pickle.dump(refs, f)

print(f'Aligned records: {len(before_preds)}')
print(f'Saved to {OUT_DIR}/')

Aligned records: 2199
Saved to /content/eval_pkls/


In [ ]:
# Cell 7: full evaluation using exact metrics.py functions
import subprocess
subprocess.run(['pip', 'install', '-q', 'evaluate', 'bert_score', 'openai'])

# ── API CONFIG ─────────────────────────────────────────────────────────────────
OPENAI_API_KEY = ''  # paste TAMU AI Chat API key here
BASE_URL       = 'https://chat-api.tamu.ai/api'  # TAMU AI Chat API Endpoint

GPT_SAMPLE = 300
# ───────────────────────────────────────────────────────────────────────────────

import importlib.util, os, sys, shutil, random, httpx
import numpy as np

# Set up evaluation/ directory so metrics.py can load system_prompt.txt
os.makedirs('/content/evaluation', exist_ok=True)
shutil.copy('/content/system_prompt.txt', '/content/evaluation/system_prompt.txt')
os.chdir('/content')
sys.path.insert(0, '/content')

# Patch metrics.py to use the TAMU AI model instead of standard OpenAI models
with open('/content/metrics.py', 'r') as f:
    metrics_code = f.read()
metrics_code = metrics_code.replace('"gpt-3.5-turbo"', '"protected.gpt-4o"')
metrics_code = metrics_code.replace("'gpt-3.5-turbo'", '"protected.gpt-4o"')
metrics_code = metrics_code.replace('"gpt-4"', '"protected.gpt-4o"')
metrics_code = metrics_code.replace("'gpt-4'", '"protected.gpt-4o"')
with open('/content/metrics.py', 'w') as f:
    f.write(metrics_code)

# Import metrics.py directly (exact original code)
spec = importlib.util.spec_from_file_location('metrics', '/content/metrics.py')
m    = importlib.util.module_from_spec(spec)

# Temporarily modify sys.argv to prevent argparse from failing on kernel arguments
_original_argv = sys.argv
sys.argv = [sys.argv[0]]
spec.loader.exec_module(m)
sys.argv = _original_argv # Restore sys.argv

# Override the OpenAI client
from openai import OpenAI
m.client = OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL)
print(f'Using API endpoint: {BASE_URL}')

# ── BERTScore (full, GPU) ──────────────────────────────────────────────────────
import evaluate as ev
bertscore = ev.load('bertscore')

print(f'BERTScore BEFORE on {len(before_preds)} records (GPU)...')
bs_b = bertscore.compute(predictions=before_preds, references=refs,
                          lang='en', rescale_with_baseline=True, device='cuda')
print(f'BERTScore AFTER on {len(after_preds)} records (GPU)...')
bs_a = bertscore.compute(predictions=after_preds, references=refs,
                          lang='en', rescale_with_baseline=True, device='cuda')

bf1  = np.mean(bs_b['f1']);  af1  = np.mean(bs_a['f1'])
bp   = np.mean(bs_b['precision']); ap  = np.mean(bs_a['precision'])
br   = np.mean(bs_b['recall']);    ar  = np.mean(bs_a['recall'])
bf1s = np.std(bs_b['f1']);   af1s = np.std(bs_a['f1'])

# ── USR (exact function from metrics.py) ──────────────────────────────────────
before_tokens = [s.split() for s in before_preds]
after_tokens  = [s.split() for s in after_preds]
usr_before, _ = m.unique_sentence_percent(before_tokens)
usr_after,  _ = m.unique_sentence_percent(after_tokens)

# ── Patch m.get_gpt_response to handle plain-string returns from TAMU API ─────
import re as _re
def _patched_gpt_response(prompt):
    try:
        completion = m.client.chat.completions.create(
            messages=[{'role': 'system', 'content': m.system_prompt},
                      {'role': 'user',   'content': prompt}],
            model='protected.gpt-4o',
        )
        raw = (completion if isinstance(completion, str)
               else completion.choices[0].message.content).strip()
        match = _re.search(r'[-+]?\d+(?:\.\d+)?', raw)
        return float(match.group()) if match else float('nan')
    except Exception:
        return float('nan')
m.get_gpt_response = _patched_gpt_response

print('BERTScore + USR computed')

Using API endpoint: https://chat-api.tamu.ai/api


BERTScore BEFORE on 2199 records (GPU)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore AFTER on 2199 records (GPU)...
BERTScore + USR computed


In [20]:
# Cell 8: Robust GPT scorer + final table
import json as _json, concurrent.futures as _cf, re as _re
import numpy as np
import random

# ── Robust GPT scorer (catches errors, reports failures, 10 workers) ──────────
_first_bad = [None]

def _safe_gpt_response(prompt):
    try:
        completion = m.client.chat.completions.create(
            messages=[{'role': 'system', 'content': m.system_prompt},
                      {'role': 'user',   'content': prompt}],
            model='protected.gpt-4o',
        )

        raw = ""
        if isinstance(completion, str):
            # Parse SSE stream
            extracted = []
            for line in completion.split('\n'):
                if line.startswith('data: ') and line.strip() != 'data: [DONE]':
                    try:
                        chunk = _json.loads(line[6:])
                        if 'choices' in chunk and len(chunk['choices']) > 0:
                            delta = chunk['choices'][0].get('delta', {})
                            if 'content' in delta:
                                extracted.append(delta['content'])
                    except Exception:
                        pass
            raw = "".join(extracted).strip()
        else:
            raw = completion.choices[0].message.content.strip()

        # Accept bare numbers OR embedded numbers (e.g. 'Score: 45', '45/100')
        m_ = _re.search(r'[-+]?\d+(?:\.\d+)?', raw)
        if m_:
            return float(m_.group())
        if _first_bad[0] is None:
            _first_bad[0] = raw
        return float('nan')
    except Exception as e:
        if _first_bad[0] is None:
            _first_bad[0] = repr(e)
        return float('nan')

def _safe_get_gpt_score(predictions, references, workers=10):
    prompts = [_json.dumps({'prediction': p, 'reference': r})
               for p, r in zip(predictions, references)]
    with _cf.ThreadPoolExecutor(max_workers=workers) as ex:
        results = list(ex.map(_safe_gpt_response, prompts))
    valid = [x for x in results if not np.isnan(x)]
    n_fail = len(results) - len(valid)
    if n_fail:
        print(f'  [WARN] {n_fail}/{len(results)} GPT calls failed (excluded from stats)')
    if not valid:
        return float('nan'), float('nan')
    return np.mean(valid), np.std(valid)

# ── GPT Score (full dataset) ───────────────────────────────────────────────────────
# Removed sampling to run on all data
s_before = before_preds
s_after  = after_preds
s_ref    = refs

print(f'\nGPT Score BEFORE on {len(s_before)} samples...')
gpt_before, gpt_before_std = _safe_get_gpt_score(s_before, s_ref)
print(f'GPT Score AFTER on {len(s_after)} samples...')
gpt_after,  gpt_after_std  = _safe_get_gpt_score(s_after,  s_ref)

# ── Final table ───────────────────────────────────────────────────────────────
print()
print('=' * 72)
print(f'  BEFORE vs AFTER FINE-TUNING  (n={len(before_preds)}, ref=ChatGPT)')
print('=' * 72)
print(f'  {"Metric":<32} {"Before":>10} {"After":>10} {"Delta":>8}')
print('  ' + '-' * 63)
print(f'  {"BERTScore F1":<32} {bf1:>10.4f} {af1:>10.4f} {af1-bf1:>+8.4f}')
print(f'  {"BERTScore Precision":<32} {bp:>10.4f} {ap:>10.4f} {ap-bp:>+8.4f}')
print(f'  {"BERTScore Recall":<32} {br:>10.4f} {ar:>10.4f} {ar-br:>+8.4f}')
print(f'  {"BERTScore F1 std":<32} {bf1s:>10.4f} {af1s:>10.4f}')
print(f'  {"GPT Score (0-100, n="+str(len(s_before))+")":<32} {gpt_before:>10.4f} {gpt_after:>10.4f} {gpt_after-gpt_before:>+8.4f}')
print(f'  {"GPT std":<32} {gpt_before_std:>10.4f} {gpt_after_std:>10.4f}')
print(f'  {"USR (diversity)":<32} {usr_before:>10.4f} {usr_after:>10.4f} {usr_after-usr_before:>+8.4f}')
print('=' * 72)
print(f'  BERTScore F1  : {"improved" if af1 > bf1 else "did not improve"} ({af1-bf1:+.4f})')
print(f'  GPT Score     : {"improved" if gpt_after > gpt_before else "did not improve"} ({gpt_after-gpt_before:+.4f})')


GPT Score BEFORE on 2199 samples...
GPT Score AFTER on 2199 samples...

  BEFORE vs AFTER FINE-TUNING  (n=2199, ref=ChatGPT)
  Metric                               Before      After    Delta
  ---------------------------------------------------------------
  BERTScore F1                         0.0154     0.4500  +0.4345
  BERTScore Precision                  0.2156     0.5418  +0.3261
  BERTScore Recall                    -0.1718     0.3598  +0.5316
  BERTScore F1 std                     0.0676     0.1148
  GPT Score (0-100, n=2199)           57.9900    69.4088 +11.4188
  GPT std                             34.8298    26.5436
  USR (diversity)                      0.9845     1.0000  +0.0155
  BERTScore F1  : improved (+0.4345)
  GPT Score     : improved (+11.4188)
